# 04 · Consultas

Responde a la pregunta del proyecto con las consultas de `sql/02_consultas.sql`.

**Pregunta:** ¿qué jugador ha metido más triples en proporción al tiempo jugado en la historia de la NBA?

**Hipótesis 1:** Stephen Curry es el jugador con más triples por cada 36 minutos jugados.

**Hipótesis 2:** al aumentar la altura se cogen más rebotes: cada franja de unos 5 cm coge más rebotes por 36 minutos que la anterior.

In [1]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# lee usuario y contraseña del archivo .env de la raíz del repo
load_dotenv("../.env")

DB_NAME = os.getenv("DB_NAME", "nba_triples")

def crear_url(base_de_datos=None):
    return URL.create(
        "mysql+pymysql",
        username=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        host=os.getenv("DB_HOST", "localhost"),
        port=int(os.getenv("DB_PORT", 3306)),
        database=base_de_datos,
        query={"charset": "utf8mb4"},
    )

engine = create_engine(crear_url(DB_NAME))

def consulta(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

## 1. Ranking de triples por cada 36 minutos (mínimo 5.000 minutos)

Exigimos un mínimo de minutos para que no salga primero alguien que jugó tres partidos.

In [2]:
consulta("""
SELECT j.nombre,
       SUM(e.minutos) AS minutos,
       SUM(e.triples) AS triples,
       ROUND(SUM(e.triples) * 36 / SUM(e.minutos), 2) AS triples_por_36
FROM estadisticas e
JOIN jugadores j ON j.player_id = e.player_id
GROUP BY j.player_id, j.nombre
HAVING SUM(e.minutos) >= 5000
ORDER BY triples_por_36 DESC
LIMIT 10
""")

,nombre,minutos,triples,triples_por_36
0,Stephen Curry,36304.0,4248.0,4.21
1,Isaiah Joe,6943.0,765.0,3.97
2,Troy Daniels,5045.0,555.0,3.96
3,Duncan Robinson,13126.0,1422.0,3.90
4,Sam Hauser,6664.0,720.0,3.89
5,Malik Beasley,13739.0,1453.0,3.81
6,Buddy Hield,20999.0,2201.0,3.77
7,LaMelo Ball,9392.0,977.0,3.74
8,Dāvis Bertāns,8546.0,885.0,3.73
9,Steve Novak,5657.0,575.0,3.66


## 2. ¿Cambia el ranking con un mínimo de 20.000 minutos? (CTE + RANK)

Si el primero sigue siendo el mismo con mínimos más altos, la conclusión no depende del corte que elijamos.

In [3]:
consulta("""
WITH carrera AS (
    SELECT player_id,
           SUM(minutos) AS minutos,
           SUM(triples) * 36 / SUM(minutos) AS triples_por_36
    FROM estadisticas
    GROUP BY player_id
    HAVING SUM(minutos) >= 20000
)
SELECT RANK() OVER (ORDER BY c.triples_por_36 DESC) AS puesto,
       j.nombre, c.minutos, ROUND(c.triples_por_36, 2) AS triples_por_36
FROM carrera c
JOIN jugadores j ON j.player_id = c.player_id
ORDER BY puesto
LIMIT 5
""")

,puesto,nombre,minutos,triples_por_36
0,1,Stephen Curry,36304.0,4.21
1,2,Buddy Hield,20999.0,3.77
2,3,Klay Thompson,29306.0,3.56
3,4,Donovan Mitchell,20585.0,3.22
4,5,Damian Lillard,32614.0,3.10


## 3. Triples totales frente a triples por minuto

Los que más triples han metido en total no son necesariamente los que más meten por minuto.

In [4]:
consulta("""
SELECT j.nombre,
       SUM(e.triples) AS triples_totales,
       ROUND(SUM(e.triples) * 36 / SUM(e.minutos), 2) AS triples_por_36
FROM estadisticas e
JOIN jugadores j ON j.player_id = e.player_id
GROUP BY j.player_id, j.nombre
ORDER BY triples_totales DESC
LIMIT 10
""")

,nombre,triples_totales,triples_por_36
0,Stephen Curry,4248.0,4.21
1,James Harden,3390.0,2.88
2,Ray Allen,2973.0,2.31
3,Klay Thompson,2899.0,3.56
4,Damian Lillard,2804.0,3.10
5,LeBron James,2636.0,1.55
6,Reggie Miller,2560.0,1.94
7,Kyle Korver,2450.0,2.83
8,Paul George,2449.0,2.79
9,Kevin Durant,2377.0,1.94


## 4. Evolución del triple por décadas

Triples por cada 36 minutos y porcentaje de acierto de toda la liga.

In [5]:
consulta("""
SELECT (temporada DIV 10) * 10 AS decada,
       ROUND(SUM(triples) * 36 / SUM(minutos), 2) AS triples_por_36,
       ROUND(SUM(triples) / SUM(triples_intentados), 3) AS acierto
FROM estadisticas
GROUP BY decada
ORDER BY decada
""")

,decada,triples_por_36,acierto
0,1980,0.15,0.289
1,1990,0.59,0.347
2,2000,0.83,0.356
3,2010,1.23,0.356
4,2020,1.91,0.361


## 5. Posiciones de los 50 mejores tiradores por minuto

¿De qué posición son los mejores tiradores?

In [6]:
consulta("""
WITH top50 AS (
    SELECT player_id,
           SUM(triples) * 36 / SUM(minutos) AS triples_por_36
    FROM estadisticas
    GROUP BY player_id
    HAVING SUM(minutos) >= 5000
    ORDER BY triples_por_36 DESC
    LIMIT 50
)
SELECT j.posicion, COUNT(*) AS jugadores
FROM top50 t
JOIN jugadores j ON j.player_id = t.player_id
GROUP BY j.posicion
ORDER BY jugadores DESC
""")

,posicion,jugadores
0,G,32
1,F,10
2,G-F,6
3,F-G,1
4,F-C,1


## 6. Hipótesis 2: ¿al aumentar la altura se cogen más rebotes?

Agrupamos a los jugadores por altura de 2 en 2 pulgadas (unos 5 cm) y calculamos sus rebotes por cada 36 minutos, con el mismo mínimo de 5.000 minutos que en la hipótesis 1.

`cambio_vs_anterior` usa `LAG()` para comparar cada franja con la anterior. Si sale positivo en todas, la hipótesis se confirma. Quitamos las franjas con menos de 10 jugadores (las alturas más extremas), porque con 1 o 4 jugadores la media depende de una sola persona.

In [7]:
consulta("""
WITH carrera AS (
    -- rebotes por 36 minutos de toda la carrera de cada jugador
    SELECT e.player_id,
           j.altura_pulgadas,
           SUM(e.rebotes) * 36 / SUM(e.minutos) AS rebotes_por_36
    FROM estadisticas e
    JOIN jugadores j ON j.player_id = e.player_id
    GROUP BY e.player_id, j.altura_pulgadas
    HAVING SUM(e.minutos) >= 5000
),
franjas AS (
    -- agrupo las alturas de 2 en 2 pulgadas (unos 5 cm)
    SELECT FLOOR(altura_pulgadas / 2) * 2 AS franja_pulgadas,
           COUNT(*) AS jugadores,
           AVG(rebotes_por_36) AS media,
           STDDEV(rebotes_por_36) AS desviacion,
           MIN(rebotes_por_36) AS minimo,
           MAX(rebotes_por_36) AS maximo
    FROM carrera
    GROUP BY franja_pulgadas
    HAVING COUNT(*) >= 10   -- quito las franjas con menos de 10 jugadores, salen medias poco fiables
)
SELECT franja_pulgadas,
       ROUND(franja_pulgadas * 2.54) AS desde_cm,
       jugadores,
       ROUND(media, 2) AS rebotes_por_36,
       ROUND(desviacion, 2) AS desviacion,
       ROUND(minimo, 2) AS minimo,
       ROUND(maximo, 2) AS maximo,
       -- LAG mira la franja anterior: si sale positivo, esta franja coge más rebotes
       ROUND(media - LAG(media) OVER (ORDER BY franja_pulgadas), 2) AS cambio_vs_anterior
FROM franjas
ORDER BY franja_pulgadas
""")

,franja_pulgadas,desde_cm,jugadores,rebotes_por_36,desviacion,minimo,maximo,cambio_vs_anterior
0,70,178.0,14,3.11,0.51,2.31,3.86,NaN
1,72,183.0,92,3.46,0.71,2.21,5.48,0.34
2,74,188.0,172,3.59,0.83,1.89,7.16,0.13
3,76,193.0,212,4.40,1.15,2.23,8.33,0.81
4,78,198.0,302,5.70,1.77,2.36,14.92,1.30
5,80,203.0,308,7.90,2.05,3.35,13.33,2.20
6,82,208.0,229,9.12,1.97,3.76,16.28,1.22
7,84,213.0,89,9.59,1.88,5.35,15.72,0.47
8,86,218.0,14,9.78,1.47,7.72,13.17,0.19


## Conclusiones

- Stephen Curry es el primero en triples por cada 36 minutos con cualquiera de los mínimos de minutos que hemos probado. La hipótesis se confirma.
- Detrás de él el ranking cambia según el mínimo: con pocos minutos aparecen especialistas que jugaron poco; con muchos, tiradores con carreras largas.
- El triple ha crecido muchísimo desde los años 80, así que los jugadores recientes salen favorecidos.
- Son relaciones entre datos, no causas.
- Hipótesis 2: se confirma. Cuanto más alto es el jugador, más rebotes coge por cada 36 minutos: de unos 3,1 en los que miden alrededor de 1,78 m a casi 9,8 en los de 2,18 m.
- El salto grande está entre 1,98 y 2,08 m. A partir de 2,13 m sigue subiendo, pero muy poco: medir más ayuda cada vez menos.
- Dentro de cada franja hay mucha diferencia entre jugadores (mira la desviación y el máximo), así que la altura no lo explica todo. También cuentan la posición y el papel de cada jugador en su equipo.